<h1 style="text-align:center; color:green; font-size:48px;">
DHNx PROJECT
</h1>

# Import libraries

In [ ]:
import matplotlib.pyplot as plt
import dhnx
import pandas as pd
import oemof.solph
from pyomo.environ import SolverFactory

# Example 1 

## 1.1 Create network and plot

In [ ]:
# Initialize thermal network
network = dhnx.network.ThermalNetwork()

# Load town parameter
network = network.from_csv_folder(r"DHNx_files/Step2/twn_data")

# Load investment parameter
invest_opt = dhnx.input_output.load_invest_options(r"DHNx_files/Step2/invest_data")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import contextily as ctx

sns.set_theme(style="white", context="talk")


static_map = dhnx.plotting.StaticMap(network)
fig, ax = static_map.draw(background_map=False)

for coll in ax.collections:
    try:
        coll.set_linewidth(5)          # thicker pipes
        coll.set_color("#0099FF")      # or e.g. "steelblue", "#2E86C1"
        coll.set_alpha(0.9)            # optional transparency
    except Exception:
        pass  # skip non-line collections

# --- High resolution and figure size
fig.set_size_inches(10, 10)
fig.set_dpi(50)

# --- Bounds with margin
lon = pd.concat([
    network.components.consumers['lon'],
    network.components.producers['lon'],
    network.components.forks['lon'],
])
lat = pd.concat([
    network.components.consumers['lat'],
    network.components.producers['lat'],
    network.components.forks['lat'],
])
expand = 0.25
ax.set_xlim(lon.min() - expand*(lon.max()-lon.min()), lon.max() + expand*(lon.max()-lon.min()))
ax.set_ylim(lat.min() - expand*(lat.max()-lat.min()), lat.max() + expand*(lat.max()-lat.min()))

# --- FREE basemap (no API key)
ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.Esri.WorldStreetMap)

# --- Overlays (nodes)
palette = sns.color_palette("Set2", 3)
ax.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
           color=palette[0], label='Consumers', zorder=6, s=200, edgecolor='k')
ax.scatter(network.components.producers['lon'], network.components.producers['lat'],
           color=palette[1], label='Producers', zorder=6, s=200, edgecolor='k')
ax.scatter(network.components.forks['lon'], network.components.forks['lat'],
           color=palette[2], label='Forks', zorder=6, s=200, edgecolor='k')

# --- Cosmetics
ax.set_title('DH Network', fontsize=28, fontweight='bold')
ax.grid(False)
ax.tick_params(axis='both', labelsize=20)
ax.set_xlabel("Longitude", fontsize=18)
ax.set_ylabel("Latitude", fontsize=18)
ax.legend(loc='lower center', ncol=3, frameon=True, fontsize=20)
sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.show()

# --- Optional: save high-resolution image
# fig.savefig("DH_network_highres.png", dpi=600, bbox_inches='tight')


## 1.2 Investment optimization of the network

In [ ]:
# Optimize the investment data
network.optimize_investment(invest_options=invest_opt,  solver='cbc')


## 1.3 Results postprocessing and Plotting 


In [ ]:
# ####### Postprocessing and Plotting ###########
results_edges = network.results.optimization['components']['pipes']
print(results_edges[['from_node', 'to_node', 'hp_type', 'capacity',
                     'direction', 'costs', 'losses']])

results_edges.to_csv("Outputs/Ex1.results_edges.csv", index=True)

print('Objective value: ', network.results.optimization['oemof_meta']['objective'])

# assign new ThermalNetwork with invested pipes
twn_results = network
twn_results.components['pipes'] = results_edges[results_edges['capacity'] > 0.001]

# # plot invested edges
# static_map_2 = dhnx.plotting.StaticMap(twn_results)
# static_map_2.draw(background_map=False)
# plt.title('Result network')
# plt.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
#             color='tab:green', label='consumers', zorder=2.5, s=50)
# plt.scatter(network.components.producers['lon'], network.components.producers['lat'],
#             color='tab:red', label='producers', zorder=2.5, s=50)
# plt.scatter(network.components.forks['lon'], network.components.forks['lat'],
#             color='tab:grey', label='forks', zorder=2.5, s=50)
# plt.text(-2, 32, 'P0', fontsize=14)
# plt.text(82, 0, 'P1', fontsize=14)
# plt.legend()
# plt.show()

static_map = dhnx.plotting.StaticMap(twn_results)
fig, ax = static_map.draw(background_map=False)

for coll in ax.collections:
    try:
        coll.set_linewidth(5)          # thicker pipes
        coll.set_color("#0099FF")      # or e.g. "steelblue", "#2E86C1"
        coll.set_alpha(0.9)            # optional transparency
    except Exception:
        pass  # skip non-line collections

# --- High resolution and figure size
fig.set_size_inches(20, 20)
fig.set_dpi(300)

# --- Bounds with margin
lon = pd.concat([
    network.components.consumers['lon'],
    network.components.producers['lon'],
    network.components.forks['lon'],
])
lat = pd.concat([
    network.components.consumers['lat'],
    network.components.producers['lat'],
    network.components.forks['lat'],
])
expand = 0.25
ax.set_xlim(lon.min() - expand*(lon.max()-lon.min()), lon.max() + expand*(lon.max()-lon.min()))
ax.set_ylim(lat.min() - expand*(lat.max()-lat.min()), lat.max() + expand*(lat.max()-lat.min()))

# --- FREE basemap (no API key)
ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.Esri.WorldStreetMap)

# --- Overlays (nodes)
palette = sns.color_palette("Set2", 3)
ax.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
           color=palette[0], label='Consumers', zorder=6, s=200, edgecolor='k')
ax.scatter(network.components.producers['lon'], network.components.producers['lat'],
           color=palette[1], label='Producers', zorder=6, s=200, edgecolor='k')
ax.scatter(network.components.forks['lon'], network.components.forks['lat'],
           color=palette[2], label='Forks', zorder=6, s=200, edgecolor='k')

# --- Cosmetics
ax.set_title('DH Network', fontsize=28, fontweight='bold')
ax.grid(False)
ax.tick_params(axis='both', labelsize=20)
ax.set_xlabel("Longitude", fontsize=18)
ax.set_ylabel("Latitude", fontsize=18)
ax.legend(loc='lower center', ncol=3, frameon=True, fontsize=20)
sns.despine(left=False, bottom=False)

plt.tight_layout()
plt.show()

# Example 2

## 2.1 Create and visualize the thermal network

In [ ]:
# Initialize thermal network
network = dhnx.network.ThermalNetwork()

# Load town data file for town parameters
network = network.from_csv_folder(r"DHNx_files/Step2/twn_data")

# Load investment parameter
invest_opt = dhnx.input_output.load_invest_options(r"DHNx_files/Step2/invest_data")

In [ ]:
# Draw network
static_map = dhnx.plotting.StaticMap(network)
static_map.draw(background_map=False)
plt.title("DH network")
plt.show()

## 2.2 investment optimization of the thermal network

In [ ]:
# Execute investment optimization
network.optimize_investment(invest_options=invest_opt, write_lp_file=True, solver='cbc')

## 2.3 Postprocessing and Plotting

In [ ]:
# get results
results_edges = network.results.optimization["components"]["pipes"]
print("*Results*")
print(results_edges)
print("")
print("Objective Value: ", network.results.optimization["oemof_meta"]["objective"])

# manually recalculate total costs
total_costs = (33 * 3.162 + 15 * 1 + 18 * 1 + 18 * 0.5) * 0.5
print("Costs re-calculation: ", total_costs)

# get indices which are existing or invested
ind = results_edges[results_edges["capacity"] > 0].index

# select invested edges
network_result = network
network_result.components["pipes"] = results_edges.loc[ind]

# Plotting the optimization result
static_map = dhnx.plotting.StaticMap(network)
static_map.draw(background_map=False)
plt.title('Optimization result')
plt.scatter(network.components.consumers['lon'], network.components.consumers['lat'],
            color='tab:green', label='consumers', zorder=2.5, s=50)
plt.scatter(network.components.producers['lon'], network.components.producers['lat'],
            color='tab:red', label='producers', zorder=2.5, s=50)
plt.scatter(network.components.forks['lon'], network.components.forks['lat'],
            color='tab:grey', label='forks', zorder=2.5, s=50)
plt.legend()
plt.show()